# PCD Assignment 03 - Image Segmentation and Morphological Processing

## Otsu Thresholding Segmentation with Morphological Operations (Without Library)

This project implements one image segmentation method, **Otsu Thresholding**, and several **morphological operations** (Erosion, Dilation, Opening, and Closing) completely from scratch, without using any image processing library such as OpenCV, scikit-image, or SciPy.

The segmentation result without morphological processing is then compared with the result after morphological processing, both visually and numerically (number of objects and number of noise blobs).

Only basic libraries are used: `PIL` to read and save the image file, `numpy` as an array container, and `matplotlib` to display the results. All processing steps are written manually.

## 0. Import Libraries

`PIL` is used only to read the input image file and `matplotlib` only to display it. No image processing function from any library is used.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from google.colab import drive, files

## 1. Load the Original Image

The input image is uploaded to Colab and loaded from `/content`. Run the cell below and choose your image file when the upload button appears.

If no image is uploaded, a synthetic noisy test image is generated so the notebook can still be run.

In [ ]:
uploaded = files.upload()

image_path = None
if len(uploaded) > 0:
    image_path = '/content/' + list(uploaded.keys())[0]

if image_path is not None and os.path.exists(image_path):
    image_rgb = np.array(Image.open(image_path).convert('RGB'), dtype=np.uint8)
    print("Image loaded successfully:", image_path, image_rgb.shape)
else:
    print("Image not found, generating a synthetic noisy test image.")
    np.random.seed(0)
    H, W = 200, 200
    yy, xx = np.mgrid[0:H, 0:W]
    g = np.full((H, W), 60.0)
    for cy, cx, r in [(50, 50, 28), (60, 140, 22), (140, 70, 30), (150, 150, 25)]:
        g[(yy - cy) ** 2 + (xx - cx) ** 2 <= r ** 2] = 190.0
    g += np.random.normal(0, 28, (H, W))
    sp = np.random.rand(H, W)
    g[sp < 0.02] = 255
    g[sp > 0.98] = 0
    g = np.clip(g, 0, 255).astype(np.uint8)
    image_rgb = np.stack([g, g, g], axis=2)

## 2. Original Image

The original image is displayed in full color before any processing is applied.

In [ ]:
plt.figure(figsize=(8, 6))
plt.imshow(image_rgb)
plt.title("Original Image (RGB)")
plt.axis("off")
plt.show()

## 3. Convert to Grayscale (Manual)

Segmentation by thresholding works on a single intensity channel, so the color image is converted to grayscale manually using the weighted formula `gray = 0.299 R + 0.587 G + 0.114 B`.

Because the morphological operations and the labeling step below are written in plain Python, very large images are subsampled (every k-th pixel is kept) so that the notebook still runs quickly.

In [ ]:
def to_gray(rgb):
    r = rgb[:, :, 0].astype(np.float64)
    g = rgb[:, :, 1].astype(np.float64)
    b = rgb[:, :, 2].astype(np.float64)
    return np.floor(0.299 * r + 0.587 * g + 0.114 * b + 0.5).astype(np.uint8)

gray = to_gray(image_rgb)

MAX_SIDE = 400
step = int(np.ceil(max(gray.shape) / MAX_SIDE))
if step > 1:
    gray = gray[::step, ::step]

print("Working size:", gray.shape)

plt.figure(figsize=(8, 6))
plt.imshow(gray, cmap='gray')
plt.title("Grayscale Image")
plt.axis("off")
plt.show()

## 4. Histogram of the Original Image

A histogram represents the distribution of pixel intensity values. It is computed manually by counting how many pixels have each intensity from 0 to 255.

For an image with a clear object and background, the histogram shows two groups of intensities, which is the situation where Otsu Thresholding works well.

In [ ]:
def histogram(img):
    h = np.zeros(256, dtype=np.int64)
    for v in img.ravel():
        h[v] += 1
    return h

hist = histogram(gray)

plt.figure(figsize=(10, 5))
plt.bar(range(256), hist, width=1.0)
plt.title("Histogram of Grayscale Image")
plt.xlabel("Pixel Intensity")
plt.ylabel("Number of Pixels")
plt.show()

## 5. Segmentation: Otsu Thresholding

Otsu Thresholding automatically finds the best threshold `t` that separates the image into background and foreground. It tests every possible threshold and chooses the one that maximizes the between-class variance:

`variance_between(t) = w0(t) * w1(t) * (mean0(t) - mean1(t))^2`

where `w0` and `w1` are the proportions of pixels in the two classes and `mean0` and `mean1` are their average intensities. Pixels brighter than the threshold become foreground (1), and the rest become background (0).

If the objects you want to segment are darker than the background, set `INVERT = True`.

In [ ]:
def otsu_threshold(img, hist):
    prob = hist / img.size
    sum_all = 0.0
    for i in range(256):
        sum_all += i * prob[i]

    w0, sum0 = 0.0, 0.0
    best_t, best_var = 0, -1.0
    for t in range(256):
        w0 += prob[t]
        sum0 += t * prob[t]
        w1 = 1.0 - w0
        if w0 <= 0 or w1 <= 0:
            continue
        m0 = sum0 / w0
        m1 = (sum_all - sum0) / w1
        var_between = w0 * w1 * (m0 - m1) ** 2
        if var_between > best_var:
            best_var, best_t = var_between, t
    return best_t

T = otsu_threshold(gray, hist)
print("Otsu threshold =", T)

INVERT = False
binary = (gray > T).astype(np.uint8)
if INVERT:
    binary = 1 - binary

### Histogram with Otsu Threshold and Segmentation Result (Without Morphology)

The red line marks the threshold found by Otsu's method. The binary image is the segmentation result **without** any morphological processing.

In [ ]:
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.bar(range(256), hist, width=1.0)
plt.axvline(T, color='red')
plt.title("Histogram with Otsu Threshold (T = %d)" % T)
plt.xlabel("Pixel Intensity")
plt.ylabel("Number of Pixels")

plt.subplot(1, 2, 2)
plt.imshow(binary, cmap='gray')
plt.title("Segmentation Without Morphology")
plt.axis("off")

plt.tight_layout()
plt.show()

## 6. Morphological Operations

Four morphological operations are implemented manually on the binary image using a structuring element (SE), a small shape that slides over the image:

- **Erosion**: a pixel stays 1 only if every pixel under the SE is 1. It shrinks objects and removes small white noise.
- **Dilation**: a pixel becomes 1 if any pixel under the SE is 1. It enlarges objects and fills small holes.
- **Opening**: erosion followed by dilation. It removes small white noise while keeping the size of large objects.
- **Closing**: dilation followed by erosion. It fills small black holes and gaps inside objects.

The SE is symmetric, so no reflection is needed, and the image border is padded with 0.

In [ ]:
def make_se(size=3, shape='square'):
    se = np.zeros((size, size), dtype=np.uint8)
    c = size // 2
    for i in range(size):
        for j in range(size):
            if shape == 'square':
                se[i, j] = 1
            elif shape == 'disk' and (i - c) ** 2 + (j - c) ** 2 <= c * c:
                se[i, j] = 1
            elif shape == 'cross' and (i == c or j == c):
                se[i, j] = 1
    return se

def pad_zero(img, ph, pw):
    H, W = img.shape
    out = np.zeros((H + 2 * ph, W + 2 * pw), dtype=img.dtype)
    out[ph:ph + H, pw:pw + W] = img
    return out

def erosion(img, se):
    H, W = img.shape
    kh, kw = se.shape
    p = pad_zero(img, kh // 2, kw // 2)
    out = np.ones((H, W), dtype=np.uint8)
    for i in range(kh):
        for j in range(kw):
            if se[i, j] == 1:
                out = out & p[i:i + H, j:j + W]
    return out

def dilation(img, se):
    H, W = img.shape
    kh, kw = se.shape
    p = pad_zero(img, kh // 2, kw // 2)
    out = np.zeros((H, W), dtype=np.uint8)
    for i in range(kh):
        for j in range(kw):
            if se[i, j] == 1:
                out = out | p[i:i + H, j:j + W]
    return out

def opening(img, se):
    return dilation(erosion(img, se), se)

def closing(img, se):
    return erosion(dilation(img, se), se)

# Try make_se(5, 'disk') if the image is very noisy
SE = make_se(3, 'disk')
print("Structuring element:")
print(SE)

eroded = erosion(binary, SE)
dilated = dilation(binary, SE)
opened = opening(binary, SE)
closed = closing(binary, SE)
cleaned = closing(opened, SE)

### Result of Each Morphological Operation

The four basic operations are displayed side by side. `cleaned` is opening followed by closing, which is used as the final segmentation result **with** morphological processing.

In [ ]:
plt.figure(figsize=(16, 4.5))

titles = ["Erosion", "Dilation", "Opening", "Closing"]
results = [eroded, dilated, opened, closed]
for i in range(4):
    plt.subplot(1, 4, i + 1)
    plt.imshow(results[i], cmap='gray')
    plt.title(titles[i])
    plt.axis("off")

plt.tight_layout()
plt.show()

## 7. Comparison of Segmentation Without and With Morphology

The segmentation result using only Otsu Thresholding is compared with the result after applying opening followed by closing.

In [ ]:
plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
plt.imshow(binary, cmap='gray')
plt.title("Without Morphology (Otsu Only)")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(cleaned, cmap='gray')
plt.title("With Morphology (Opening + Closing)")
plt.axis("off")

plt.tight_layout()
plt.show()

## 8. Numerical Comparison

To measure the effect of morphological processing, connected components (8-connectivity) are counted manually. Components smaller than `NOISE_AREA` pixels are counted as noise, and the remaining ones are counted as real objects.

In [ ]:
def component_sizes(img):
    H, W = img.shape
    visited = np.zeros((H, W), dtype=bool)
    sizes = []
    for y in range(H):
        for x in range(W):
            if img[y, x] == 1 and not visited[y, x]:
                stack = [(y, x)]
                visited[y, x] = True
                count = 0
                while stack:
                    cy, cx = stack.pop()
                    count += 1
                    for dy in (-1, 0, 1):
                        for dx in (-1, 0, 1):
                            ny, nx = cy + dy, cx + dx
                            if 0 <= ny < H and 0 <= nx < W and img[ny, nx] == 1 and not visited[ny, nx]:
                                visited[ny, nx] = True
                                stack.append((ny, nx))
                sizes.append(count)
    return sizes

NOISE_AREA = 30

def summarize(name, img):
    s = component_sizes(img)
    noise = sum(1 for a in s if a < NOISE_AREA)
    return name, len(s), noise, len(s) - noise, int(img.sum())

rows = [
    summarize('Otsu only (no morphology)', binary),
    summarize('Otsu + opening', opened),
    summarize('Otsu + closing', closed),
    summarize('Otsu + opening + closing', cleaned),
]

print('%-28s %10s %10s %10s %14s' % ('Result', 'components', 'noise', 'objects', 'white pixels'))
for r in rows:
    print('%-28s %10d %10d %10d %14d' % r)

## 9. Save the Results

The original, segmentation without morphology, and segmentation with morphology images are saved to the PCD_Assignment03 folder in Google Drive. These files can later be uploaded to the GitHub repository.

In [ ]:
drive.mount('/content/drive')

output_path = '/content/drive/MyDrive/PCD_Assignment03/'
os.makedirs(output_path, exist_ok=True)

Image.fromarray(image_rgb).save(output_path + 'original_rgb.jpg')
Image.fromarray(binary * 255).save(output_path + 'segmentation_without_morphology.png')
Image.fromarray(cleaned * 255).save(output_path + 'segmentation_with_morphology.png')

print("Images saved successfully.")

## 10. Observation

The segmentation using only Otsu Thresholding separates the objects from the background, but every pixel is classified independently. Because of this, noise in the image produces many small isolated white blobs in the background and small black holes inside the objects, which makes the number of detected components much larger than the number of real objects.

Opening removes the small isolated white blobs while keeping the size of the large objects. Closing fills the small holes and gaps inside objects, but on its own it does not remove the isolated noise blobs. Applying opening followed by closing gives the cleanest result, where the noise components are almost completely removed and the real objects stay intact. This is visible in the side-by-side comparison in Section 7 and in the component counts in Section 8.

A larger structuring element removes more noise, but it also erodes thin or small real structures and rounds off their corners. If too much noise remains, `SE` in Section 6 can be enlarged (e.g. `make_se(5, 'disk')`); if real objects are being damaged, it can be reduced.

## 11. Conclusion

In this experiment, Otsu Thresholding was implemented from scratch to segment an image, and four morphological operations (Erosion, Dilation, Opening, and Closing) were implemented from scratch to improve the segmentation result, without using any image processing library.

The comparison shows that Otsu Thresholding alone produces a noisy segmentation with many small false components and holes. After applying opening followed by closing, the noise was removed and the objects became solid and cleaner, so the number of detected components matched the number of real objects much more closely.

This shows that morphological processing is an effective post-processing step for thresholding-based segmentation, as long as the structuring element is chosen small enough to preserve the real objects in the image.